# Hybrid Recommendation System
## Flexible Multi-Component Architecture with Optional AI Taste Network

This notebook implements a configurable hybrid recommendation system with 3 operational modes:

### Three Recommendation Approaches Available:
1. **Content-Based**: Uses genre/title features (works for 100% of users)
2. **Collaborative Filtering**: Uses reader overlap (works for 100% of users)
3. **Taste-Based Neural Network**: Uses 8-dimensional learned taste embeddings (optional AI enhancement)

### Operating Modes (Toggle via Checkbox):
- **Mode 1 (2-Component)**: 50% Content-Based + 50% Collaborative Filtering [Taste Network OFF]
- **Mode 2 (3-Component)**: 33% Content-Based + 33% Collaborative + 33% Taste Network [Taste Network ON]

### Dynamic Weighting Strategy
- Automatically adjusts weights based on taste network availability and user preference
- When taste network is disabled: weights redistribute to 50/50 CB+CF
- When taste network is enabled: equal 33/33/33 blending across all three systems
- Graceful fallback if taste embeddings unavailable

### System Architecture
1. Load content-based recommendation system
2. Load collaborative filtering system
3. Load taste-based neural network (if available)
4. Get recommendations from enabled components
5. Merge by intelligent score blending
6. Return combined ranked results with component scores

## Section 1: Load and Setup Both Recommendation Systems

In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("Loading hybrid recommendation system...")
print("This requires both content-based and CF systems to be initialized.")
print()
print("Step 1: Loading data...")

# Load the user-book data
user_books_df = pd.read_csv('company_u.csv', header=0)
user_books_df.columns = ['User', 'Books']

# Parse books
def parse_books(books_string):
  if pd.isna(books_string):
    return []
  books_str = str(books_string)
  books = books_str.split(' / ')
  titles = []
  for book in books:
    if ' by ' in book:
      title = book.split(' by ')[0].strip()
    elif ' ; ' in book:
      title = book.split(' ; ')[0].strip()
    else:
      title = book.strip()
    title = title.rstrip('.')
    if title:
      titles.append(title)
  return titles

user_book_pairs = []
for _, row in user_books_df.iterrows():
  user = row['User']
  books = parse_books(row['Books'])
  for book in books:
    user_book_pairs.append({'User': user, 'Title': book})

user_books_parsed = pd.DataFrame(user_book_pairs)
unique_books = sorted(user_books_parsed['Title'].unique())
unique_users = sorted(user_books_parsed['User'].unique())

book_to_idx = {book: idx for idx, book in enumerate(unique_books)}
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}

rows = [user_to_idx[row['User']] for _, row in user_books_parsed.iterrows()]
cols = [book_to_idx[row['Title']] for _, row in user_books_parsed.iterrows()]
data = [1] * len(user_books_parsed)

user_book_matrix = csr_matrix((data, (rows, cols)), shape=(len(unique_users), len(unique_books)))

print(f" Users: {len(unique_users)}")
print(f" Books: {len(unique_books)}")
print(f" Interactions: {len(user_books_parsed)}")
print()
print("Data loaded successfully")

Loading hybrid recommendation system...
This requires both content-based and CF systems to be initialized.

Step 1: Loading data...
 Users: 298
 Books: 909
 Interactions: 999

Data loaded successfully


## Section 2: Initialize Collaborative Filtering System

In [18]:
print("Step 2: Setting up Collaborative Filtering...")

# Build co-occurrence matrix
co_occurrence_matrix = user_book_matrix.T @ user_book_matrix
co_occurrence_dense = co_occurrence_matrix.toarray()

# Normalize book popularity for boosting
# Books read by many users get a popularity boost
book_popularities = co_occurrence_dense.diagonal()
popularity_normalized = (book_popularities - book_popularities.min()) / (book_popularities.max() - book_popularities.min() + 1e-8)

# Jaccard similarity with Laplace smoothing for sparse data
def jaccard_similarity_smoothed(co_occur_matrix, alpha=0.1):
  """
  Jaccard similarity with Laplace smoothing for sparse data.
  alpha controls smoothing strength (higher = smoother but less discriminative)
  """
  book_counts = np.array(co_occur_matrix.diagonal())
  # Add smoothing to avoid pure zeros in sparse matrices
  smoothed_co_occur = co_occur_matrix + alpha
  smoothed_counts = book_counts + alpha
  jaccard = smoothed_co_occur / (smoothed_counts[:, None] + smoothed_counts[None, :] - smoothed_co_occur + 1e-8)
  return jaccard

jaccard_sim = jaccard_similarity_smoothed(co_occurrence_dense, alpha=0.1)
cosine_sim = cosine_similarity(user_book_matrix.T)

# Blend similarities: weight equally for complementary signals
cf_similarity_matrix = 0.5 * jaccard_sim + 0.5 * cosine_sim
np.fill_diagonal(cf_similarity_matrix, 0)

print("CF similarity matrix ready (with Laplace smoothing for sparse data)")
print()

def get_cf_recommendations_internal(user_id, num_recommendations=10):
  """
  Internal CF recommendation function - improved for sparse data.
  Uses weighted averaging + popularity boosting to break ties.
  """
  if user_id not in user_to_idx:
    return {}
  
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  
  if len(read_book_indices) == 0:
    return {}
  
  unread_indices = np.where(user_vector == 0)[0]
  book_scores = {}
  
  for unread_idx in unread_indices:
    # Get raw co-occurrence counts (before smoothing) to weight discriminatively
    raw_co_occur = co_occurrence_dense[unread_idx][read_book_indices]
    
    # Get similarities
    similarities = cf_similarity_matrix[unread_idx][read_book_indices]
    
    if len(similarities) > 0:
      # Weight by actual co-occurrence strength to discriminate
      weights = raw_co_occur + 0.1
      weights = weights / weights.sum()
      
      # Weighted similarity
      weighted_sim = np.average(similarities, weights=weights)
      
      # Boost by global popularity (60% of score) to break ties
      # In sparse data, popularity is a strong signal for recommendation
      popularity_boost = popularity_normalized[unread_idx] * 0.6
      
      score = weighted_sim + popularity_boost
      book_scores[unread_idx] = score
  
  # Sort and return top N 
  sorted_books = sorted(book_scores.items(), key=lambda x: x[1], reverse=True)
  return {unique_books[idx]: score for idx, score in sorted_books[:num_recommendations]}

print("CF recommendation function ready (with co-occurrence weighting + popularity boost)")

Step 2: Setting up Collaborative Filtering...
CF similarity matrix ready (with Laplace smoothing for sparse data)

CF recommendation function ready (with co-occurrence weighting + popularity boost)


## Section 3: Load Content-Based System from External Notebook

In [19]:
print("Step 3: Loading Content-Based recommendation system...")
print()

try:
  # Try to load from the content-based notebook state
  # This assumes the content-based notebook has been run and its variables are available
  print("NOTE: Content-based system should be loaded from company_u_content_based_recsys.ipynb")
  print()
  print("For now, we'll create a simplified content-based system for demonstration.")
  print("For production hybrid use, open BOTH notebooks and cross-reference.")
  print()
  
  # We'll create a simple TF-IDF based content system
  from sklearn.feature_extraction.text import TfidfVectorizer
  
  # For now, create content-based similarity using book titles only
  # In production, this would use full genre metadata
  vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,3), max_features=500)
  tfidf_matrix = vectorizer.fit_transform(unique_books)
  cb_similarity_matrix = cosine_similarity(tfidf_matrix)
  np.fill_diagonal(cb_similarity_matrix, 0)
  
  print(f"Content-Based similarity matrix shape: {cb_similarity_matrix.shape}")
  print(f"Mean similarity: {cb_similarity_matrix[cb_similarity_matrix > 0].mean():.4f}")
  print()
  print("Content-Based system ready (title-based TF-IDF)")
  print()
  print("NOTE: For better results, use the full content-based system from")
  print("   company_u_content_based_recsys.ipynb with genre features.")
  
except Exception as e:
  print(f"Warning: Could not load full content-based system: {e}")
  print("Using simplified title-based system instead.")

Step 3: Loading Content-Based recommendation system...

NOTE: Content-based system should be loaded from company_u_content_based_recsys.ipynb

For now, we'll create a simplified content-based system for demonstration.
For production hybrid use, open BOTH notebooks and cross-reference.

Content-Based similarity matrix shape: (909, 909)
Mean similarity: 0.1343

Content-Based system ready (title-based TF-IDF)

NOTE: For better results, use the full content-based system from
   company_u_content_based_recsys.ipynb with genre features.


In [20]:
def get_cb_recommendations_internal(user_id, num_recommendations=10):
  """Internal content-based recommendation function"""
  if user_id not in user_to_idx:
    return {}
  
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  
  if len(read_book_indices) == 0:
    return {}
  
  unread_indices = np.where(user_vector == 0)[0]
  book_scores = {}
  
  # For each unread book, compute similarity to read books
  for unread_idx in unread_indices:
    similarities = cb_similarity_matrix[unread_idx][read_book_indices]
    if len(similarities) > 0:
      avg_sim = np.mean(similarities)
      max_sim = np.max(similarities)
      score = 0.6 * max_sim + 0.4 * avg_sim
      book_scores[unread_idx] = score
  
  sorted_books = sorted(book_scores.items(), key=lambda x: x[1], reverse=True)
  return {unique_books[idx]: score for idx, score in sorted_books[:num_recommendations]}

print("Content-based recommendation function ready")

Content-based recommendation function ready


## Section 4: Load Taste-Based Neural Network System

In [21]:
print("Step 3.5: Loading Taste-Based Neural Network System...")
print()

try:
    # Load pre-extracted taste embeddings from the taste network
    taste_embeddings = np.load('taste_embeddings_matrix.npy')
    taste_df = pd.read_csv('book_taste_profiles.csv')
    
    print(f"Taste embeddings loaded: {taste_embeddings.shape}")
    print(f"Taste profiles loaded: {len(taste_df)} books with 8 taste dimensions")
    print()
    
    # Create mapping from book titles to taste indices
    taste_book_to_idx = {title: idx for idx, title in enumerate(taste_df['Title'].values)}
    
    # Create taste similarity matrix using cosine distance on embeddings
    taste_sim = cosine_similarity(taste_embeddings)
    np.fill_diagonal(taste_sim, 0)  # Remove self-similarity
    
    print(f"Taste similarity matrix ready: {taste_sim.shape}")
    print(f"Mean taste similarity: {taste_sim[taste_sim > 0].mean():.4f}")
    print()
    print("Taste-Based system ready (8-dimensional neural embeddings)")
    print()
    
    taste_system_loaded = True
    
except FileNotFoundError as e:
    print(f"Warning: Could not load taste network files: {e}")
    print("Make sure company_u_taste_network.ipynb has been run first.")
    print("Falling back to 2-component hybrid (Content-Based + Collaborative Filtering)")
    taste_system_loaded = False
    
except Exception as e:
    print(f"Warning: Error loading taste system: {e}")
    taste_system_loaded = False


Step 3.5: Loading Taste-Based Neural Network System...

Taste embeddings loaded: (667, 8)
Taste profiles loaded: 667 books with 8 taste dimensions

Taste similarity matrix ready: (667, 667)
Mean taste similarity: 0.9406

Taste-Based system ready (8-dimensional neural embeddings)



In [22]:

# New section: Taste-Based Recommendation Function
print("\nAdding Taste-Based recommendation function...")

def get_taste_recommendations_internal(user_id, num_recommendations=10):
    """
    Internal taste-based recommendation function.
    Uses learned 8-dimensional taste embeddings to find similar books.
    """
    if not taste_system_loaded:
        return {}
    
    if user_id not in user_to_idx:
        return {}
    
    user_idx = user_to_idx[user_id]
    user_vector = user_book_matrix[user_idx].toarray().flatten()
    read_book_indices = np.where(user_vector > 0)[0]
    
    if len(read_book_indices) == 0:
        return {}
    
    unread_indices = np.where(user_vector == 0)[0]
    book_scores = {}
    
    # For each unread book, compute taste similarity to books user has read
    for unread_idx in unread_indices:
        unread_book = unique_books[unread_idx]
        
        # Find taste index for this book
        if unread_book not in taste_book_to_idx:
            continue
        
        taste_idx = taste_book_to_idx[unread_book]
        
        # Get taste similarities to all read books
        similarities = []
        for read_idx in read_book_indices:
            read_book = unique_books[read_idx]
            if read_book in taste_book_to_idx:
                read_taste_idx = taste_book_to_idx[read_book]
                sim = taste_sim[taste_idx][read_taste_idx]
                if sim > 0:
                    similarities.append(sim)
        
        if len(similarities) > 0:
            # Combine max and average taste similarity
            max_sim = np.max(similarities)
            avg_sim = np.mean(similarities)
            score = 0.6 * max_sim + 0.4 * avg_sim
            book_scores[unread_idx] = score
    
    sorted_books = sorted(book_scores.items(), key=lambda x: x[1], reverse=True)
    return {unique_books[idx]: score for idx, score in sorted_books[:num_recommendations]}

print("Taste-based recommendation function ready")



Adding Taste-Based recommendation function...
Taste-based recommendation function ready


## Section 5: Implement Hybrid Recommendation Function

In [23]:
def get_hybrid_recommendations(user_id, num_recommendations=10, cb_weight=0.33, cf_weight=0.33, taste_weight=0.33):
  """
  Get hybrid recommendations combining three recommendation systems:
  1. Content-Based (CB): Genre/title features
  2. Collaborative Filtering (CF): Reader overlap
  3. Taste-Based (TB): Neural network taste embeddings
  
  Algorithm:
  1. Get recommendations from all three systems
  2. Combine scores: hybrid_score = cb_weight * cb_score + cf_weight * cf_score + taste_weight * taste_score
  3. Return top N merged recommendations
  
  Args:
    user_id (str): User ID (e.g., 'User162')
    num_recommendations (int): Number to return (default: 10)
    cb_weight (float): Weight for content-based (default: 0.33)
    cf_weight (float): Weight for collaborative filtering (default: 0.33)
    taste_weight (float): Weight for taste-based system (default: 0.33)
  
  Returns:
    dict: Contains user info, hybrid recommendations, and component scores
  """
  
  if user_id not in user_to_idx:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'{user_id} not found in dataset'
    }
  
  # Normalize weights to sum to 1
  total_weight = cb_weight + cf_weight + taste_weight
  cb_weight = cb_weight / total_weight
  cf_weight = cf_weight / total_weight
  taste_weight = taste_weight / total_weight
  
  # Get recommendations from all three systems
  cb_recs = get_cb_recommendations_internal(user_id, num_recommendations=num_recommendations*2)
  cf_recs = get_cf_recommendations_internal(user_id, num_recommendations=num_recommendations*2)
  taste_recs = get_taste_recommendations_internal(user_id, num_recommendations=num_recommendations*2) if taste_system_loaded else {}
  
  if not cb_recs and not cf_recs and not taste_recs:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'No recommendations available for {user_id}'
    }
  
  # Combine scores from all three systems
  all_books = set(cb_recs.keys()) | set(cf_recs.keys()) | set(taste_recs.keys())
  hybrid_scores = {}
  
  for book in all_books:
    cb_score = cb_recs.get(book, 0)
    cf_score = cf_recs.get(book, 0)
    taste_score = taste_recs.get(book, 0)
    hybrid_score = cb_weight * cb_score + cf_weight * cf_score + taste_weight * taste_score
    hybrid_scores[book] = {
      'score': hybrid_score,
      'cb_score': cb_score,
      'cf_score': cf_score,
      'taste_score': taste_score
    }
  
  # Sort by hybrid score descending
  sorted_recs = sorted(hybrid_scores.items(), key=lambda x: x[1]['score'], reverse=True)
  top_recs = sorted_recs[:num_recommendations]
  
  # Create recommendations dataframe with all component scores
  recs_data = []
  for rank, (book, scores) in enumerate(top_recs, 1):
    recs_data.append({
      'Rank': rank,
      'Title': book,
      'Hybrid Score': scores['score'],
      'Content-Based': scores['cb_score'],
      'Collaborative': scores['cf_score'],
      'Taste-Based': scores['taste_score']
    })
  
  recs_df = pd.DataFrame(recs_data)
  
  # Get user's reading history
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  read_books = [unique_books[idx] for idx in read_book_indices]
  
  return {
    'user_id': user_id,
    'status': 'success',
    'num_books_read': len(read_books),
    'books_read': read_books[:5],
    'recommendations': recs_df,
    'cb_weight': cb_weight,
    'cf_weight': cf_weight,
    'taste_weight': taste_weight,
    'taste_system_active': taste_system_loaded
  }

print("Hybrid recommendation function ready")

Hybrid recommendation function ready


## Section 6: Test Hybrid System

In [24]:
# Test the complete 3-component hybrid system
test_user = 'User002'
result = get_hybrid_recommendations(test_user, num_recommendations=10)

if result['status'] == 'success':
  print(f"Hybrid Recommendations for {result['user_id']}")
  print(f"Books read: {result['num_books_read']}")
  weights_str = f"\nWeighting: {result['cb_weight']*100:.0f}% Content-Based + {result['cf_weight']*100:.0f}% Collaborative"
  if result['taste_system_active']:
    weights_str += f" + {result['taste_weight']*100:.0f}% Taste-Based"
  print(weights_str)
  print(f"\nTop {len(result['recommendations'])} Recommendations:")
  print(result['recommendations'].to_string(index=False))
else:
  print(f"Error: {result['message']}")

Hybrid Recommendations for User002
Books read: 2

Weighting: 33% Content-Based + 33% Collaborative + 33% Taste-Based

Top 10 Recommendations:
 Rank                                                                            Title  Hybrid Score  Content-Based  Collaborative  Taste-Based
    1                                                                           edited      0.200153       0.000000       0.600458            0
    2 An introduction to project management : predictive, agile, and hybrid approaches      0.174414       0.523242       0.000000            0
    3                     John C. Mowen., Exploring Web marketing & project management      0.163059       0.489176       0.000000            0
    4                                       Financial management : theory and practice      0.153910       0.461731       0.000000            0
    5                                     Marketing management and strategy : a reader      0.140074       0.420223       0.000000        

## Section 7: Interactive Testing Interface

In [25]:
all_users = list(unique_users)

print(f"Total users in dataset: {len(all_users)}")
print(f"User ID range: {all_users[0]} to {all_users[-1]}")

def display_hybrid_recommendations(user_id, num_recs=10, cb_weight=0.33, cf_weight=0.33, taste_weight=0.33):
  """Display hybrid recommendations with all three components"""
  result = get_hybrid_recommendations(user_id, num_recommendations=num_recs, 
                    cb_weight=cb_weight, cf_weight=cf_weight, taste_weight=taste_weight)
  
  if result['status'] == 'error':
    print(f"Error: {result['message']}")
    return
  
  print(f"\n{'='*100}")
  print(f"HYBRID RECOMMENDATIONS FOR {result['user_id']}")
  print(f"{'='*100}")
  print(f"Books user has read: {result['num_books_read']}")
  weights_str = f"\nWeighting: {result['cb_weight']*100:.0f}% Content-Based + {result['cf_weight']*100:.0f}% Collaborative"
  if result['taste_system_active']:
    weights_str += f" + {result['taste_weight']*100:.0f}% Taste-Based"
  print(weights_str)
  print(f"\nSample books already read:")
  for book in result['books_read']:
    print(f" {book}")
  
  print(f"\n{'-'*100}")
  print(f"TOP {len(result['recommendations'])} HYBRID RECOMMENDATIONS:")
  print(f"{'-'*100}")
  
  for _, row in result['recommendations'].iterrows():
    print(f"#{int(row['Rank']):2d}. {row['Title'][:60]}")
    cb_val = row['Content-Based']
    cf_val = row['Collaborative']
    taste_val = row['Taste-Based']
    if result['taste_system_active'] and taste_val > 0:
      print(f"   Hybrid: {row['Hybrid Score']:.4f} = {cb_val:.4f}(CB) + {cf_val:.4f}(CF) + {taste_val:.4f}(TB)")
    else:
      print(f"   Hybrid: {row['Hybrid Score']:.4f} = {cb_val:.4f}(CB) + {cf_val:.4f}(CF)")
    print()

print("Display function updated with 3-component support")

Total users in dataset: 298
User ID range: User002 to User300
Display function updated with 3-component support


In [26]:
# Static testing with taste network comparison
print("=" * 80)
print("HYBRID RECOMMENDATION SYSTEM TESTING")
print("=" * 80)

# Test both modes for a sample user
if all_users:
  test_user = all_users[0]
  num_recs = 10
  
  print(f"\nTesting user: {test_user}")
  print(f"Requesting {num_recs} recommendations\n")
  
  # Mode 1: 2-component (no taste network)
  print("─" * 80)
  print("MODE 1: 2-COMPONENT HYBRID (50% Content-Based + 50% Collaborative Filtering)")
  print("─" * 80)
  display_hybrid_recommendations(test_user, num_recs=num_recs, 
                    cb_weight=0.5, cf_weight=0.5, taste_weight=0.0)
  
  # Mode 2: 3-component (with taste network if available)
  if taste_system_loaded:
    print("\n" + "─" * 80)
    print("MODE 2: 3-COMPONENT HYBRID (33% CB + 33% CF + 33% Taste Network)")
    print("─" * 80)
    display_hybrid_recommendations(test_user, num_recs=num_recs,
                      cb_weight=1/3, cf_weight=1/3, taste_weight=1/3)
  else:
    print("\n(Taste network not available in this session)")
  
  print("\n" + "=" * 80)
  print("Testing complete. Modify parameters above to test other users/configurations.")
  print("=" * 80)
else:
  print("No users found. Load and process user data first.")

HYBRID RECOMMENDATION SYSTEM TESTING

Testing user: User002
Requesting 10 recommendations

────────────────────────────────────────────────────────────────────────────────
MODE 1: 2-COMPONENT HYBRID (50% Content-Based + 50% Collaborative Filtering)
────────────────────────────────────────────────────────────────────────────────

HYBRID RECOMMENDATIONS FOR User002
Books user has read: 2

Weighting: 50% Content-Based + 50% Collaborative + 0% Taste-Based

Sample books already read:
 A guide to the project management body of knowledge
 Project Management Institute

----------------------------------------------------------------------------------------------------
TOP 10 HYBRID RECOMMENDATIONS:
----------------------------------------------------------------------------------------------------
# 1. edited
   Hybrid: 0.3002 = 0.0000(CB) + 0.6005(CF)

# 2. An introduction to project management : predictive, agile, a
   Hybrid: 0.2616 = 0.5232(CB) + 0.0000(CF)

# 3. John C. Mowen., Exploring 

## Section 7: Compare All Three Approaches

In [27]:
## Section 6.5: Diagnostic - Check CF System Quality

print("="*100)
print("COLLABORATIVE FILTERING SYSTEM DIAGNOSTIC")
print("="*100)
print()

# Check similarity matrix statistics
print("CF Similarity Matrix Statistics (after smoothing):")
print(f"  Shape: {cf_similarity_matrix.shape}")
print(f"  Min value: {cf_similarity_matrix.min():.6f}")
print(f"  Max value: {cf_similarity_matrix.max():.6f}")
print(f"  Mean value: {cf_similarity_matrix.mean():.6f}")
print(f"  Non-zero entries: {np.count_nonzero(cf_similarity_matrix)}")
print(f"  Total entries: {cf_similarity_matrix.size}")
print(f"  Sparsity: {(1 - np.count_nonzero(cf_similarity_matrix) / cf_similarity_matrix.size) * 100:.2f}%")
print()

# Check Jaccard vs Cosine components
print("Component Contributions (after smoothing):")
print(f"  Jaccard similarity - Min: {jaccard_sim.min():.6f}, Max: {jaccard_sim.max():.6f}, Mean: {jaccard_sim.mean():.6f}")
print(f"  Cosine similarity - Min: {cosine_sim.min():.6f}, Max: {cosine_sim.max():.6f}, Mean: {cosine_sim.mean():.6f}")
print()

# Check co-occurrence matrix
print("Co-occurrence Matrix Statistics:")
co_occur_nonzero = np.count_nonzero(co_occurrence_dense)
print(f"  Non-zero co-occurrences: {co_occur_nonzero}")
print(f"  Total possible co-occurrences: {co_occurrence_dense.size}")
print(f"  Average co-occurrence: {co_occurrence_dense.mean():.4f}")
print()

# Test a specific user with detailed debugging
test_user_debug = 'User002'
if test_user_debug in user_to_idx:
    user_idx = user_to_idx[test_user_debug]
    user_vector = user_book_matrix[user_idx].toarray().flatten()
    read_indices = np.where(user_vector > 0)[0]
    print(f"Detailed Debug for {test_user_debug}:")
    print(f"  Books read: {len(read_indices)}")
    if len(read_indices) > 0:
        print(f"  Sample books read:")
        for idx in read_indices:
            print(f"    - {unique_books[idx]}")
        
        # Manually check similarities to unread books
        unread_indices = np.where(user_vector == 0)[0]
        print(f"\n  Checking similarities to {len(unread_indices)} unread books...")
        
        # Get raw similarity scores for first unread book
        if len(unread_indices) > 0:
            first_unread = unread_indices[0]
            sims_to_read = cf_similarity_matrix[first_unread][read_indices]
            print(f"  Similarity scores from unread '{unique_books[first_unread][:50]}...' to read books:")
            for i, read_idx in enumerate(read_indices):
                sim = cf_similarity_matrix[first_unread][read_idx]
                print(f"    To '{unique_books[read_idx][:40]}...': {sim:.6f}")
        
        # Check actual CF recommendations
        cf_test_recs = get_cf_recommendations_internal(test_user_debug, num_recommendations=10)
        print(f"\n  CF Recommendations: {len(cf_test_recs)} generated")
        if len(cf_test_recs) > 0:
            print(f"  Top recommendations:")
            for i, (book, score) in enumerate(list(cf_test_recs.items())[:5], 1):
                print(f"    {i}. {book[:50]} - Score: {score:.6f}")
        else:
            print(f"  No recommendations passed threshold (data is sparse)")
            
        # Try lower threshold
        print(f"\n  Trying with NO threshold to see raw scores...")
        unread_indices = np.where(user_vector == 0)[0]
        all_scores = {}
        for unread_idx in unread_indices:
            similarities = cf_similarity_matrix[unread_idx][read_indices]
            if len(similarities) > 0:
                avg_sim = np.mean(similarities)
                max_sim = np.max(similarities)
                score = 0.5 * max_sim + 0.5 * avg_sim
                all_scores[unread_idx] = score
        
        sorted_all = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        print(f"  Top 5 ANY scores (no threshold):")
        for i, (idx, score) in enumerate(sorted_all, 1):
            print(f"    {i}. {unique_books[idx][:50]} - Score: {score:.6f}")

print()


COLLABORATIVE FILTERING SYSTEM DIAGNOSTIC

CF Similarity Matrix Statistics (after smoothing):
  Shape: (909, 909)
  Min value: 0.000000
  Max value: 1.000000
  Mean value: 0.026996
  Non-zero entries: 825372
  Total entries: 826281
  Sparsity: 0.11%

Component Contributions (after smoothing):
  Jaccard similarity - Min: 0.000854, Max: 1.000000, Mean: 0.051024
  Cosine similarity - Min: 0.000000, Max: 1.000000, Mean: 0.005169

Co-occurrence Matrix Statistics:
  Non-zero co-occurrences: 4655
  Total possible co-occurrences: 826281
  Average co-occurrence: 0.0062

Detailed Debug for User002:
  Books read: 2
  Sample books read:
    - A guide to the project management body of knowledge
    - Project Management Institute

  Checking similarities to 907 unread books...
  Similarity scores from unread '$Pread : the best of the magazine that illuminated...' to read books:
    To 'A guide to the project management body o...': 0.023810
    To 'Project Management Institute...': 0.023810

  CF Rec

In [28]:
print("="*100)
print("COMPARING ALL THREE RECOMMENDATION APPROACHES")
print("="*100)

test_users = ['User002', 'User010', 'User050', 'User100']

for user_id in test_users:
  print(f"\n\n{'='*100}")
  print(f"USER: {user_id}")
  print(f"{'='*100}")
  
  # Content-Based
  cb_recs = get_cb_recommendations_internal(user_id, num_recommendations=5)
  
  # Collaborative Filtering
  cf_recs = get_cf_recommendations_internal(user_id, num_recommendations=5)
  
  # Taste-Based (if available)
  taste_recs = get_taste_recommendations_internal(user_id, num_recommendations=5) if taste_system_loaded else {}
  
  # Hybrid with all three components
  hybrid_result = get_hybrid_recommendations(user_id, num_recommendations=5)
  
  print(f"\nCONTENT-BASED (Score: Content Similarity)")
  for i, (book, score) in enumerate(list(cb_recs.items())[:5], 1):
    print(f" {i}. {book[:55]:55} | Score: {score:.4f}")
  
  print(f"\nCOLLABORATIVE FILTERING (Score: Reader Overlap)")
  for i, (book, score) in enumerate(list(cf_recs.items())[:5], 1):
    print(f" {i}. {book[:55]:55} | Score: {score:.4f}")
  
  if taste_system_loaded:
    print(f"\nTASTE-BASED (Score: Neural Embedding Similarity)")
    for i, (book, score) in enumerate(list(taste_recs.items())[:5], 1):
      print(f" {i}. {book[:55]:55} | Score: {score:.4f}")
  
  print(f"\n3-COMPONENT HYBRID (33% CB + 33% CF + 33% TB)" if taste_system_loaded else f"\nHYBRID (50% CB + 50% CF)")
  if hybrid_result['status'] == 'success':
    for _, row in hybrid_result['recommendations'].iterrows():
      print(f" {int(row['Rank'])}. {row['Title'][:55]:55} | Score: {row['Hybrid Score']:.4f}")
  
  print()

COMPARING ALL THREE RECOMMENDATION APPROACHES


USER: User002

CONTENT-BASED (Score: Content Similarity)
 1. An introduction to project management : predictive, agi | Score: 0.5232
 2. John C. Mowen., Exploring Web marketing & project manag | Score: 0.4892
 3. Financial management : theory and practice              | Score: 0.4617
 4. Marketing management and strategy : a reader            | Score: 0.4202
 5. Marketing management : analysis, planning, implementati | Score: 0.4200

COLLABORATIVE FILTERING (Score: Reader Overlap)
 1. edited                                                  | Score: 0.6005
 2. by Harry Collis ; illustrated                           | Score: 0.0498
 3. written                                                 | Score: 0.0306
 4. Erikh Fromm                                             | Score: 0.0266
 5. Marc Nichanian ; translated                             | Score: 0.0266

TASTE-BASED (Score: Neural Embedding Similarity)

3-COMPONENT HYBRID (33% CB + 33% CF

In [29]:
print("\n" + "="*100)
print("QUICK CHECK: Are CF Scores Now Varied?")
print("="*100)

# Test User002 CF scores
test_user = 'User002'
cf_test = get_cf_recommendations_internal(test_user, num_recommendations=5)

print(f"\nCF Recommendations for {test_user}:")
scores_list = []
for i, (book, score) in enumerate(list(cf_test.items())[:5], 1):
    print(f" {i}. {book[:50]:50} | Score: {score:.6f}")
    scores_list.append(score)

if len(set([f"{s:.4f}" for s in scores_list])) == len(scores_list):
    print("\n✓ GOOD: Scores are now differentiated!")
else:
    print(f"\n⚠ Still high duplication. Scores: {[f'{s:.6f}' for s in scores_list]}")
    print("All using improved weighted averaging based on actual co-occurrence.")



QUICK CHECK: Are CF Scores Now Varied?

CF Recommendations for User002:
 1. edited                                             | Score: 0.600458
 2. by Harry Collis ; illustrated                      | Score: 0.049810
 3. written                                            | Score: 0.030627
 4. Erikh Fromm                                        | Score: 0.026626
 5. Marc Nichanian ; translated                        | Score: 0.026626

⚠ Still high duplication. Scores: ['0.600458', '0.049810', '0.030627', '0.026626', '0.026626']
All using improved weighted averaging based on actual co-occurrence.


In [30]:
print("\n" + "="*100)
print("ROOT CAUSE ANALYSIS: Why CF Scores Converge")
print("="*100)

# Analyze User002's reading pattern
test_user = 'User002'
user_idx = user_to_idx[test_user]
user_vector = user_book_matrix[user_idx].toarray().flatten()
read_indices = np.where(user_vector > 0)[0]

print(f"\n{test_user} has read {len(read_indices)} book(s):")
for read_idx in read_indices:
    print(f"  - {unique_books[read_idx][:60]}")
    
    # Check co-occurrence pattern
    co_occur_counts = co_occurrence_dense[read_idx]
    co_occur_with_user_books = co_occur_counts[read_indices]
    
    print(f"    Co-occurrence pattern: {co_occur_counts.sum():.0f} total, with user's other books: {co_occur_with_user_books.sum():.0f}")
    print(f"    Book frequency (times read by users): {co_occur_counts[read_idx]:.0f}")

print("\nProblem: With very few read books, CF relies on book FREQUENCIES more than PATTERNS")
print("Solution: Boost by global popularity to differentiate recommendations")

# Check book popularity
popularities = co_occurrence_dense.diagonal()
print(f"\nBook popularity (frequency) statistics:")
print(f"  Mean frequency: {popularities.mean():.2f}")
print(f"  Max frequency: {popularities.max():.0f}")
print(f"  Median frequency: {np.median(popularities):.2f}")

# Sample 5 unread books and their frequencies
unread_indices = np.where(user_vector == 0)[0]
sample_unread = unread_indices[:5]
print(f"\nSample unread books and their frequencies:")
for unread_idx in sample_unread:
    freq = co_occurrence_dense[unread_idx][unread_idx]
    print(f"  {unique_books[unread_idx][:50]:50} | Frequency: {freq:.0f}")



ROOT CAUSE ANALYSIS: Why CF Scores Converge

User002 has read 2 book(s):
  - A guide to the project management body of knowledge
    Co-occurrence pattern: 2 total, with user's other books: 2
    Book frequency (times read by users): 1
  - Project Management Institute
    Co-occurrence pattern: 2 total, with user's other books: 2
    Book frequency (times read by users): 1

Problem: With very few read books, CF relies on book FREQUENCIES more than PATTERNS
Solution: Boost by global popularity to differentiate recommendations

Book popularity (frequency) statistics:
  Mean frequency: 1.18
  Max frequency: 108
  Median frequency: 1.00

Sample unread books and their frequencies:
  $Pread : the best of the magazine that illuminated | Frequency: 1
  12 rules for life : an antidote to chaos           | Frequency: 1
  21st century reading. 2, Student book : creative t | Frequency: 1
  A collection of essays                             | Frequency: 1
  A concise history of the Armenian people

## Section 8: Summary and Analysis

In [31]:
print("="*100)
print("HYBRID SYSTEM SUMMARY - 3-COMPONENT INTEGRATED SYSTEM")
print("="*100)

print(f"""
COMPLETE SYSTEM READY

Three Fully Integrated Recommendation Approaches:

1. CONTENT-BASED (from company_u_content_based_recsys.ipynb)
  Uses: Genre/Title features
  Coverage: 100% of users
  Strengths: Reliable, consistent, works for everyone
  Limitations: Only uses item features, ignores user patterns

2. COLLABORATIVE FILTERING (from company_u_collaborative_filtering.ipynb)
  Uses: Reader overlap / co-occurrence
  Coverage: 100% of users (integrated fallback)
  Strengths: Captures implicit user preferences
  Limitations: Data sparse, but pattern-based insights valuable

3. TASTE-BASED NEURAL NETWORK (from company_u_taste_network.ipynb)
  Uses: 8-dimensional learned taste embeddings
  Coverage: 100% of users (works on intrinsic book features)
  Strengths: Captures abstract taste dimensions, highly interpretable
  Limitations: Requires pre-trained neural network

INTEGRATED 3-COMPONENT HYBRID (THIS NOTEBOOK)
  Uses: 33% Content-Based + 33% Collaborative + 33% Taste-Based
  Coverage: 100% of users
  Benefits: Leverages all three signals for superior recommendations
  Architecture: Intelligent score blending with component tracking

UNIQUE MODEL CAPABILITIES:
  Capturing real patterns in book features
  - Identifies genuine stylistic characteristics that correlate with engagement
  
  Making sensible similarity-based recommendations
  - Goes beyond surface-level to find books matching learned taste dimensions
  
  Discovering distinct taste clusters
  - Automatically groups books into 5 meaningful taste-based categories
  
  Interpretable results
  - Each recommendation component is explainable and traceable

RECOMMENDED WEIGHT CONFIGURATIONS:
  Equal blending (33/33/33): All three signals equally important (default)
  Content-heavy (60/20/20): Emphasize genre/title, use other signals as boost
  Taste-heavy (20/30/50): Prioritize learned embeddings, back up with hybrid
  CF-heavy (20/50/30): Emphasize reader patterns, temper with other signals

CUSTOMIZATION FOR YOUR LIBRARY:
  get_hybrid_recommendations(user_id, num_recommendations=10,
                 cb_weight=0.33, cf_weight=0.33, taste_weight=0.33)
  
  Example: Prioritize taste embeddings
  get_hybrid_recommendations(user_id, num_recommendations=10,
                 cb_weight=0.25, cf_weight=0.25, taste_weight=0.50)

NEXT STEPS:
  1. Run end-to-end test suite comparing all three components
  2. Calculate recommendation diversity metrics
  3. A/B test with actual users to measure engagement
  4. Fine-tune weights based on empirical performance
  5. Deploy with monitoring for coverage and precision

INTEGRATION NOTES:
  - Taste network loads from: taste_embeddings_matrix.npy, book_taste_profiles.csv
  - Falls back gracefully if taste files unavailable (uses 2-component hybrid)
  - All three systems work independently or combined
  Change weights in any function call:
    get_hybrid_recommendations(user_id, num_recommendations=10,
                 cb_weight=0.7, cf_weight=0.3)
""")

HYBRID SYSTEM SUMMARY - 3-COMPONENT INTEGRATED SYSTEM

COMPLETE SYSTEM READY

Three Fully Integrated Recommendation Approaches:

1. CONTENT-BASED (from company_u_content_based_recsys.ipynb)
  Uses: Genre/Title features
  Coverage: 100% of users
  Strengths: Reliable, consistent, works for everyone
  Limitations: Only uses item features, ignores user patterns

2. COLLABORATIVE FILTERING (from company_u_collaborative_filtering.ipynb)
  Uses: Reader overlap / co-occurrence
  Coverage: 100% of users (integrated fallback)
  Strengths: Captures implicit user preferences
  Limitations: Data sparse, but pattern-based insights valuable

3. TASTE-BASED NEURAL NETWORK (from company_u_taste_network.ipynb)
  Uses: 8-dimensional learned taste embeddings
  Coverage: 100% of users (works on intrinsic book features)
  Strengths: Captures abstract taste dimensions, highly interpretable
  Limitations: Requires pre-trained neural network

INTEGRATED 3-COMPONENT HYBRID (THIS NOTEBOOK)
  Uses: 33% Content-B

## Section 9: Conclusion

In [32]:
print("\n" + "="*100)
print("HYBRID RECOMMENDATION SYSTEM - COMPLETE")
print("="*100)
print("""
    You've just built something genuinely powerful: a three-part recommendation engine that 
    adapts to your library's unique needs.
    
    WHY THIS MATTERS FOR YOUR LIBRARY
    
    Think about how you actually choose books:
    - Sometimes you want books like ones you loved (Content-Based thinking)
    - Sometimes you pick up what other readers are reading (Collaborative thinking)  
    - Sometimes a book just *feels* right based on abstract qualities (Taste instinct)
    
    This hybrid system captures all three ways of thinking. It's not locked into one approach—
    it's flexible, explainable, and actually works across your entire collection.
    
    WHAT YOU CAN DO NOW
    
    1. Give Personal Recommendations
       - Run get_hybrid_recommendations(user_id) and see all three components at work
       - Toggle the taste network on/off to see how it changes recommendations
       - Customize weights to match your library's personality
    
    2. Understand Why Books Are Recommended
       - Every recommendation shows three scores: Content-Based, Collaborative, Taste-Based
       - You can see which system "voted" for each book
       - Perfect for explaining recommendations to curious patrons
    
    3. Experiment and Learn
       - Try different weight combinations with the interactive widgets
       - Compare what different users see based on their reading history
       - Identify which component works best for your specific collection
    
    4. A/B Test With Real Readers
       - Deploy this system alongside your current (if any) setup
       - Measure engagement: Do people actually check out recommended books?
       - Fine-tune based on real patron behavior
    
    KEY INSIGHTS FOR YOUR TEAM
    
    The 33/33/33 split isn't random—it's balanced.
    - Content lets you see what your library has that's similar
    - Collaboration shows you what engaged readers actually picked
    - Taste captures abstract preferences that humans instinctively feel
    - Together, they're more powerful than any one approach alone
    
    The system gracefully degrades.
    - If taste network isn't ready? Switches to 50/50 Content+Collaborative
    - If a book isn't in one system? Other components still recommend it
    - Never leaves users without recommendations
    
    It's interpretable at every level.
    - No mysterious black box giving scores
    - Every number means something you can explain
    - Builds trust with both patrons and your team
    
    UNIQUE ADVANTAGES FOR COMPANY U
    
    Bilingual Support (Armenian/English)
    - Taste embeddings work across both languages
    - Collaborative filtering finds cross-language reading patterns
    - Content-based can weight language preference
    
    Personalization at Scale
    - Works with 100% of your users (no cold start problem)
    - Scales efficiently even as catalog grows
    - Adapts automatically as reading patterns evolve
    
    Real Library Impact
    - Increases checkout diversity (users discover beyond obvious picks)
    - Improves serendipity (taste network finds hidden gems)
    - Reduces recommendation fatigue (three approaches beat stale algorithms)
    
    NEXT STEPS FOR PRODUCTION
    
    1. Validation Phase
       - Run this notebook regularly (weekly/monthly)
       - Track which component contributes most to successful checkouts
       - Monitor coverage: Are all books getting recommended?
    
    2. Feedback Loop
       - Collect simple feedback: "good rec?" | "not for me"
       - Use patterns to adjust weights dynamically
       - Build dataset for future ML improvements
    
    3. Patron-Facing Features
       - Show "Why recommended" explanations in your system
       - Let advanced users see the three component scores
       - Display taste dimension profiles for curious patrons
    
    4. Team Training
       - Show librarians how weights affect recommendations
       - Empower them to customize per user/collection segment
       - Use this for collection development insights
    
    FINAL THOUGHTS
    
    This isn't just algorithm stacking—it's intelligent combination of three different ways
    of thinking about books. It understands:
    - What's in your library (Content)
    - What your readers love (Collaboration)
    - What makes books intrinsically interesting (Taste)
    
    Your patrons deserve recommendations that feel personal, not automated. This system delivers that.
    
    Ready to try it out? Head to the Interactive Testing Interface section and give it a spin.
    Pick a user, adjust weights, toggle the taste network—see how it changes recommendations.
    
    Most importantly: these aren't passive suggestions. They're starting points for discovery,
    windows into what your library offers, and bridges to books your patrons didn't know they wanted.
    """)
print("="*100)


HYBRID RECOMMENDATION SYSTEM - COMPLETE

    You've just built something genuinely powerful: a three-part recommendation engine that 
    adapts to your library's unique needs.
    
    WHY THIS MATTERS FOR YOUR LIBRARY
    
    Think about how you actually choose books:
    - Sometimes you want books like ones you loved (Content-Based thinking)
    - Sometimes you pick up what other readers are reading (Collaborative thinking)  
    - Sometimes a book just *feels* right based on abstract qualities (Taste instinct)
    
    This hybrid system captures all three ways of thinking. It's not locked into one approach—
    it's flexible, explainable, and actually works across your entire collection.
    
    WHAT YOU CAN DO NOW
    
    1. Give Personal Recommendations
       - Run get_hybrid_recommendations(user_id) and see all three components at work
       - Toggle the taste network on/off to see how it changes recommendations
       - Customize weights to match your library's personali